# Week 3: Train Contrastive Probe from Labeled Data

## Purpose

This notebook loads the **manually labeled** training data and trains the contrastive probe.

## Input

`training_data_labeled.csv` with columns:
- `prompt`: Base prompt
- `next_token`: Actual token model predicts
- `full_text`: prompt + next_token
- `label`: **'code'** or **'language'** (manually labeled)

## Process

1. Load labeled data
2. Extract hidden states for each `full_text`
3. Train logistic regression probe
4. Test with contrastive generation

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate scipy scikit-learn pandas matplotlib seaborn

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import entropy as scipy_entropy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports complete")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    output_hidden_states=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

In [ ]:
# Cell 4: Load labeled data

LABELED_DATA_FILE = 'training_data_labeled.csv'

print(f"Loading {LABELED_DATA_FILE}...")
df = pd.read_csv(LABELED_DATA_FILE)

print(f"\n✅ Loaded {len(df)} training examples")
print(f"\nDataset info:")
print(df.info())

# Check labels
print(f"\nLabel distribution:")
print(df['label'].value_counts())

# Validate labels
valid_labels = df['label'].isin(['code', 'language'])
if not valid_labels.all():
    print(f"\n⚠️  WARNING: Found {(~valid_labels).sum()} rows with invalid labels!")
    print(f"Invalid labels: {df[~valid_labels]['label'].unique()}")
    print(f"\nRemoving invalid rows...")
    df = df[valid_labels]
    print(f"Remaining: {len(df)} examples")

# Convert to binary labels
df['label_binary'] = df['label'].map({'language': 0, 'code': 1})

print(f"\nBinary label distribution:")
print(f"  LANGUAGE (0): {(df['label_binary']==0).sum()}")
print(f"  CODE (1): {(df['label_binary']==1).sum()}")

print(f"\nSample entries:")
print(df[['full_text', 'label', 'probability']].head(10))

In [ ]:
# Cell 5: Extract hidden states

SELECTED_LAYERS = [8, 16, 31]

def get_multi_layer_state(text: str, layers: List[int]) -> np.ndarray:
    """Extract and concatenate hidden states from multiple layers."""
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    states = [
        outputs.hidden_states[layer_idx + 1][:, -1, :].cpu().numpy()[0]
        for layer_idx in layers
    ]
    return np.concatenate(states).astype(np.float32)

print(f"Extracting hidden states from layers {SELECTED_LAYERS}...")
print(f"This may take a few minutes for {len(df)} examples...\n")

X_train = []
y_train = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    h = get_multi_layer_state(row['full_text'], SELECTED_LAYERS)
    X_train.append(h)
    y_train.append(row['label_binary'])

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"\n✅ Hidden states extracted")
print(f"   Shape: {X_train.shape}")
print(f"   Feature dimension: {X_train.shape[1]:,} (3 layers × 4096)")

In [ ]:
# Cell 6: Train probe

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

probe = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(probe, X_train_scaled, y_train, cv=cv)

cv_accuracy = accuracy_score(y_train, y_pred_cv)

print(f"\n{'='*80}")
print(f"PROBE TRAINING RESULTS")
print(f"{'='*80}")
print(f"\nLayers: {SELECTED_LAYERS}")
print(f"Training examples: {len(y_train)}")
print(f"5-Fold CV Accuracy: {cv_accuracy:.1%}")

print(f"\n{classification_report(y_train, y_pred_cv, target_names=['LANGUAGE (0)', 'CODE (1)'])}")

# Confusion matrix
cm = confusion_matrix(y_train, y_pred_cv)
print(f"Confusion Matrix:")
print(f"                Pred LANG  Pred CODE")
print(f"True LANG          {cm[0,0]:>4}       {cm[0,1]:>4}")
print(f"True CODE          {cm[1,0]:>4}       {cm[1,1]:>4}")

# Train final model
probe.fit(X_train_scaled, y_train)
print(f"\n✅ Final probe trained on all data")

# Contrastive Generation Testing

In [ ]:
# Cell 7: Helper functions

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

def classify_token_type(text: str) -> Tuple[int, float]:
    """Classify: 1=code token, 0=language token."""
    h = get_multi_layer_state(text, SELECTED_LAYERS).reshape(1, -1)
    h_scaled = scaler.transform(h)
    token_type = probe.predict(h_scaled)[0]
    probability = probe.predict_proba(h_scaled)[0, 1]
    return int(token_type), float(probability)

def analyze_candidate_tokens(
    prompt: str,
    top_k: int = 10,
    verbose: bool = False
) -> Dict:
    """
    Get top-K candidate next tokens and classify each as CODE or LANGUAGE.
    Returns majority vote and detailed breakdown.
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()
    
    probs = softmax(logits)
    top_indices = np.argsort(probs)[-top_k:][::-1]
    
    candidates = []
    code_votes = 0
    lang_votes = 0
    
    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = probs[idx]
        
        # Classify prompt + candidate token
        completion = prompt + token
        token_type, type_prob = classify_token_type(completion)
        
        candidates.append({
            'token': token,
            'prob': prob,
            'type': 'CODE' if token_type == 1 else 'LANGUAGE',
            'type_prob': type_prob
        })
        
        if token_type == 1:
            code_votes += prob
        else:
            lang_votes += prob
    
    is_code_uncertainty = code_votes > lang_votes
    
    if verbose:
        print(f"\nCandidate analysis:")
        for c in candidates:
            print(f"  '{c['token']}' (p={c['prob']:.3f}) → {c['type']} (conf={c['type_prob']:.3f})")
        print(f"\nVotes: CODE={code_votes:.3f}, LANGUAGE={lang_votes:.3f}")
        print(f"Decision: {'CODE uncertainty' if is_code_uncertainty else 'LANGUAGE uncertainty'}")
    
    return {
        'candidates': candidates,
        'code_votes': code_votes,
        'lang_votes': lang_votes,
        'is_code_uncertainty': is_code_uncertainty,
        'confidence': max(code_votes, lang_votes) / (code_votes + lang_votes) if (code_votes + lang_votes) > 0 else 0
    }

print("✅ Helper functions ready")

In [ ]:
# Cell 8: Contrastive generation function

def generate_with_contrastive_probe(
    prompt: str,
    entropy_threshold: float = 3.0,
    top_k_candidates: int = 10,
    max_tokens: int = 50,
    verbose: bool = True
) -> Dict:
    """
    Contrastive entropy-driven generation.
    
    At each step:
    1. Generate next token
    2. Compute entropy H
    3. If H > threshold:
       - Get top-K candidate next tokens
       - Classify each "prompt + candidate" as CODE or LANGUAGE
       - Weighted vote: If majority CODE → STOP
       - If majority LANGUAGE → Continue
    4. If H ≤ threshold: Continue (confident)
    """
    current_text = prompt
    generated_tokens = []
    entropy_trace = []
    stop_reason = None
    stop_info = {}
    
    if verbose:
        print(f"\n{'='*80}")
        print(f"Prompt: '{prompt}'")
        print(f"Entropy threshold: {entropy_threshold:.1f} bits")
        print(f"{'='*80}")
    
    for step in range(max_tokens):
        inputs = tokenizer(current_text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits[0, -1, :].cpu().numpy()
        
        probs = softmax(logits)
        H = entropy_from_probs(probs)
        entropy_trace.append(H)
        
        next_token_id = np.argmax(probs)
        next_token = tokenizer.decode([next_token_id])
        
        if verbose:
            print(f"\nStep {step + 1}: '{next_token}' H={H:.2f}")
        
        if H > entropy_threshold:
            if verbose:
                print(f"  ⚠️  HIGH ENTROPY - analyzing candidates...")
            
            analysis = analyze_candidate_tokens(
                current_text,
                top_k=top_k_candidates,
                verbose=verbose
            )
            
            if analysis['is_code_uncertainty']:
                if verbose:
                    print(f"  ❗ CODE UNCERTAINTY - STOPPING!")
                stop_reason = "code_uncertainty"
                stop_info = {
                    'step': step,
                    'entropy': H,
                    'candidates': analysis['candidates'],
                    'code_votes': analysis['code_votes'],
                    'lang_votes': analysis['lang_votes']
                }
                break
            else:
                if verbose:
                    print(f"  ✓ LANGUAGE uncertainty - continuing")
        
        generated_tokens.append(next_token)
        current_text += next_token
        
        if next_token_id == tokenizer.eos_token_id:
            stop_reason = "eos"
            break
    
    if stop_reason is None:
        stop_reason = "max_tokens"
    
    if verbose:
        print(f"\n{'='*80}")
        print(f"Stop: {stop_reason}")
        print(f"Generated: '{current_text}'")
        print(f"{'='*80}")
    
    return {
        'prompt': prompt,
        'generated_text': ''.join(generated_tokens),
        'full_text': current_text,
        'entropy_trace': entropy_trace,
        'stop_reason': stop_reason,
        'stop_info': stop_info,
        'num_steps': len(generated_tokens)
    }

print("✅ Generation function ready")

# Testing

In [ ]:
# Cell 9: Demo test

print("\n" + "="*80)
print("DEMO: Testing Contrastive Probe")
print("="*80)

test_prompts = [
    "The authentication is done using",  # Should STOP (code uncertainty)
    "The authentication is",              # Should CONTINUE (language uncertainty)
]

for test_prompt in test_prompts:
    result = generate_with_contrastive_probe(
        test_prompt,
        entropy_threshold=3.0,
        top_k_candidates=10,
        max_tokens=5,
        verbose=True
    )
    print("\n" + "-"*80 + "\n")

In [ ]:
# Cell 10: Comprehensive testing

CODE_TEST_CASES = [
    {'prompt': 'The authentication is done using', 'expected': 1, 'category': 'auth_method'},
    {'prompt': 'We import', 'expected': 1, 'category': 'import_library'},
    {'prompt': 'The database is', 'expected': 1, 'category': 'database_type'},
    {'prompt': 'The frontend uses', 'expected': 1, 'category': 'frontend_framework'},
    {'prompt': 'Password hashing uses', 'expected': 1, 'category': 'auth_hash'},
    {'prompt': 'State management is', 'expected': 1, 'category': 'frontend_state'},
    {'prompt': 'Tokens are generated with', 'expected': 1, 'category': 'auth_token'},
    {'prompt': 'The API is built with', 'expected': 1, 'category': 'web_framework'},
    {'prompt': 'The ORM is', 'expected': 1, 'category': 'database_orm'},
    {'prompt': 'HTTP requests use', 'expected': 1, 'category': 'http_client'},
]

LANGUAGE_TEST_CASES = [
    {'prompt': 'The authentication is', 'expected': 0, 'category': 'connector_is'},
    {'prompt': 'Verification is done', 'expected': 0, 'category': 'connector_done'},
    {'prompt': 'The user id is', 'expected': 0, 'category': 'connector_is'},
    {'prompt': 'Processing is handled', 'expected': 0, 'category': 'connector_handled'},
    {'prompt': 'Data is stored', 'expected': 0, 'category': 'connector_stored'},
    {'prompt': 'The code works by', 'expected': 0, 'category': 'explanation_process'},
    {'prompt': 'The function is responsible for', 'expected': 0, 'category': 'explanation_responsibility'},
    {'prompt': 'To implement authentication,', 'expected': 0, 'category': 'instruction_to'},
    {'prompt': 'First, you need to', 'expected': 0, 'category': 'instruction_first'},
    {'prompt': 'The authentication system', 'expected': 0, 'category': 'explanation_system'},
]

ALL_TEST_CASES = CODE_TEST_CASES + LANGUAGE_TEST_CASES

print(f"\n{'='*80}")
print(f"COMPREHENSIVE TESTING")
print(f"{'='*80}")
print(f"\nTotal tests: {len(ALL_TEST_CASES)}")
print(f"  CODE tests (should stop): {len(CODE_TEST_CASES)}")
print(f"  LANGUAGE tests (should continue): {len(LANGUAGE_TEST_CASES)}")
print(f"\nRunning tests...\n")

test_results = []
for test_case in tqdm(ALL_TEST_CASES, desc="Testing"):
    result = generate_with_contrastive_probe(
        test_case['prompt'],
        entropy_threshold=3.0,
        top_k_candidates=10,
        max_tokens=5,
        verbose=False
    )
    
    predicted = 1 if result['stop_reason'] == 'code_uncertainty' else 0
    expected = test_case['expected']
    correct = predicted == expected
    
    test_results.append({
        'prompt': test_case['prompt'],
        'category': test_case['category'],
        'expected': expected,
        'predicted': predicted,
        'correct': correct,
        'stop_reason': result['stop_reason'],
        'max_entropy': np.max(result['entropy_trace']) if result['entropy_trace'] else 0,
    })

print(f"\n✅ Testing complete!")

In [ ]:
# Cell 11: Results analysis

df_results = pd.DataFrame(test_results)

print("\n" + "="*80)
print("TEST RESULTS")
print("="*80)

overall_accuracy = df_results['correct'].mean()
print(f"\n📊 OVERALL ACCURACY: {overall_accuracy:.1%}")

print(f"\n📈 BY CLASS:")
for expected_val in [1, 0]:
    class_name = "CODE (should stop)" if expected_val == 1 else "LANGUAGE (should continue)"
    subset = df_results[df_results['expected'] == expected_val]
    accuracy = subset['correct'].mean()
    correct = subset['correct'].sum()
    total = len(subset)
    print(f"\n   {class_name}")
    print(f"      Correct: {correct}/{total}")
    print(f"      Accuracy: {accuracy:.1%}")

# Confusion matrix
tp = len(df_results[(df_results['expected']==1) & (df_results['predicted']==1)])
fp = len(df_results[(df_results['expected']==0) & (df_results['predicted']==1)])
fn = len(df_results[(df_results['expected']==1) & (df_results['predicted']==0)])
tn = len(df_results[(df_results['expected']==0) & (df_results['predicted']==0)])

print(f"\n🎯 CONFUSION MATRIX")
print(f"                    Predicted LANGUAGE    Predicted CODE")
print(f"True LANGUAGE             {tn:<12}      {fp:<12}")
print(f"True CODE                 {fn:<12}      {tp:<12}")

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n📐 METRICS (CODE class)")
print(f"   Precision: {precision:.1%}")
print(f"   Recall: {recall:.1%}")
print(f"   F1-Score: {f1:.1%}")

# Errors
errors = df_results[~df_results['correct']]
if len(errors) > 0:
    print(f"\n❌ ERRORS ({len(errors)}/{len(df_results)}):")
    for _, error in errors.iterrows():
        exp = "CODE" if error['expected'] == 1 else "LANGUAGE"
        pred = "CODE" if error['predicted'] == 1 else "LANGUAGE"
        print(f"   '{error['prompt'][:50]}...'")
        print(f"      Expected: {exp}, Predicted: {pred}")
else:
    print(f"\n🎉 PERFECT! No errors!")

print(f"\n" + "="*80)

# Summary

This notebook trained a contrastive probe using **real model predictions** that were manually labeled.

Key difference from previous approach:
- **Old**: Trained on our guesses of what tokens appear (JWT, OAuth, Firebase)
- **New**: Trained on actual tokens the model predicts (O, J, the, a, `)

The probe now classifies based on the **actual distribution** of tokens the model generates, not our assumptions.